# Introduction

Let's start with the most obvious **problems** that can be seen by just looking at the database
and here are they:
#### LinkedIn
- [x] the `posted_since` column is relative to the collection date, not absolute
- [x] in the `seniority_level` column there's a value called "Not Applicable"
- [x] Extract the `salary` using a Q/A extraction model + regex
- [x] Extract the `skills` from the description
#### UpWork
- [x] There are empty strings in the `skills` column
- [x] A lot of columns have useless string components such as *"1978 <ins>hours worked</ins>"*
- [x] Money is represented using strings
- [x] Sometimes values are null and sometimes are strings indicating empty value.
#### Guru
- [x] the `earnings` and `feedback_percent` columns are represented as strings

# Setting up

In [213]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from transformers import AutoModelForQuestionAnswering, AutoTokenizer, pipeline
from datetime import datetime, timedelta, date
from typing import Optional, Tuple, List
import sqlite3
import time
import ast
import re
import os

RAW_DB_URI = "file:./data/raw_database.db?mode=ro"
CLEAN_DB_PATH = "./data/clean_database.db"
QA_MODEL_NAME     = "deepset/tinyroberta-squad2"
QA_MODEL_PATH     = "./models/tinyroberta-squad2-model"
QA_TOKENIZER_PATH = "./models/tinyroberta-squad2-tokenizer"

In [214]:
with sqlite3.connect(RAW_DB_URI, uri=True) as con:
    linkedin_df = pd.read_sql_query("SELECT * FROM linkedin", con)
    upwork_df = pd.read_sql_query("SELECT * FROM upwork", con)
    guru_df = pd.read_sql_query("SELECT * FROM guru", con)

In [215]:
linkedin_df.sample(5)

,id,posting_title,location,posted_since,company_name,description,job_url,company_url,applicants,industries,employment_type,job_function,seniority_level,searched_country,searched_job_title
986,4297226228,Applied AI Engineer Intern - Summer 2026 (May/...,"Seattle, WA",21 hours ago,Lensa,Lensa is a career site that helps job seekers ...,https://www.linkedin.com/jobs/view/applied-ai-...,https://www.linkedin.com/company/lensa?trk=pub...,Be among the first 25 applicants,Internet Publishing,Internship,Engineering and Information Technology,Internship,United States,Machine learning
186,4270270486,Data Engineer,"Aveiro, Portugal",2 weeks ago,MobiLab Solutions,Data Engineer (English speaking) (M/F/D)\nData...,https://pt.linkedin.com/jobs/view/data-enginee...,https://de.linkedin.com/company/mobilab-soluti...,None,IT Services and IT Consulting,Full-time,Information Technology,Associate,European Union,Data engineer
959,4204297989,Machine Learning Engineer - Mapping,"Foster City, CA",18 hours ago,Zoox,High-definition semantic maps are a critical t...,https://www.linkedin.com/jobs/view/machine-lea...,https://www.linkedin.com/company/zoox-inc?trk=...,Over 200 applicants,Automotive,Full-time,Engineering and Information Technology,Not Applicable,United States,Machine learning
830,4297206160,Entry-Level Data Analyst (Remote),United States,1 day ago,VroomAI,We’re looking for a Entry-Level Data Analyst w...,https://www.linkedin.com/jobs/view/entry-level...,https://www.linkedin.com/company/vroomai?trk=p...,Over 200 applicants,Data Infrastructure and Analytics,Part-time,Information Technology,Entry level,United States,Data analyst
842,4296496717,Business Data Analyst,Greater Boston,2 days ago,Digital Prospectors,Position: \nBusiness Data Analyst\nLocation:\n...,https://www.linkedin.com/jobs/view/business-da...,https://www.linkedin.com/company/digital-prosp...,Over 200 applicants,Defense and Space Manufacturing,Contract,Engineering and Analyst,Mid-Senior level,United States,Data analyst


# Data cleaning

### LinkedIn

Fixing the `posted_since` column to use dates instead of days since the data was collected<br>
NOTE: it will still be an approximatation because linkedin doesn't specify actual posting date

In [216]:
linkedin_df["posted_since"].unique()

array(['5 days ago', '7 months ago', '3 weeks ago', '2 days ago',
       '2 weeks ago', '1 week ago', '6 days ago', '4 months ago',
       '3 days ago', '4 days ago', '1 month ago', '3 months ago',
       '4 weeks ago', '2 months ago', '5 months ago', '1 day ago',
       '13 hours ago', '18 hours ago', '22 hours ago', '23 hours ago',
       '21 hours ago', '4 hours ago', '14 hours ago', '12 hours ago',
       '17 hours ago', '1 hour ago', '3 hours ago', '7 hours ago',
       '15 hours ago', '16 hours ago', '5 hours ago', '2 hours ago',
       '8 hours ago', '10 hours ago', '9 hours ago'], dtype=object)

In [217]:
collection_time = datetime.fromtimestamp(os.path.getctime("./data/linkedin_jobs.csv"))

hour_pattern  = re.compile(r"^[0-9]+ hour")
day_pattern  = re.compile(r"^[0-9]+ day")
week_pattern  = re.compile(r"^[0-9]+ week")
month_pattern = re.compile(r"^[0-9]+ month")
year_pattern  = re.compile(r"^[0-9]+ year")

value_pattern = re.compile(r"^[0-9]+")

def parse_posted_since(collection_time: datetime, posted_since: str) -> date:
    used_pattern: re.Pattern = None
    
    for pattern in [hour_pattern, day_pattern, week_pattern,
                    month_pattern, year_pattern]:
        if re.match(pattern, posted_since):
            used_pattern = pattern
            break

    if used_pattern == None:
        raise ValueError("The `posted_since` param has invalid form that can't be parsed.")

    interval_value = int(value_pattern.search(posted_since).group())
    interval_unit: timedelta = datetime.hour
    
    if used_pattern == hour_pattern:
        interval_unit = timedelta(hours=1)

    elif used_pattern == day_pattern:
        interval_unit = timedelta(days=1)
        
    elif used_pattern == week_pattern:
        interval_unit = timedelta(weeks=1)
        
    elif used_pattern == month_pattern:
        interval_unit = timedelta(days=29.53)
        
    elif  used_pattern == year_pattern:
        interval_unit = timedelta(days=365.25)

    else:
        raise ValueError("The `posted_since` param has invalid form that can't be parsed.")

    interval = interval_value * interval_unit
    
    return datetime.date(collection_time - interval)

In [218]:
linkedin_df["posted_since"] = linkedin_df["posted_since"].apply(
    lambda x: parse_posted_since(collection_time, posted_since=x))

In [219]:
linkedin_df["posted_since"].sample(5)

214    2025-09-10
483    2025-09-13
350    2025-09-12
607    2025-09-08
649    2025-09-13
Name: posted_since, dtype: object

Replacing the "Not Applicable" value in the `seniority_level` column with None

In [220]:
linkedin_df["seniority_level"].unique()

array(['Not Applicable', 'Entry level', 'Mid-Senior level', 'Associate',
       'Internship', 'Director', 'Executive'], dtype=object)

In [221]:
linkedin_df["seniority_level"] = linkedin_df["seniority_level"].replace(
    "Not Applicable", None)

Now let's try to extract the salary using an extractive Q/A model

In [222]:
if os.path.isfile(QA_MODEL_PATH) and \
   os.path.isfile(QA_TOKENIZER_PATH):
    model     = AutoModelForQuestionAnswering.from_pretrained(QA_MODEL_PATH)
    tokenizer = AutoTokenizer.from_pretrained(QA_TOKENIZER_PATH)
else:
    model     = AutoModelForQuestionAnswering.from_pretrained(QA_MODEL_NAME)
    tokenizer = AutoTokenizer.from_pretrained(QA_MODEL_NAME)
    model    .save_pretrained(QA_MODEL_PATH)
    tokenizer.save_pretrained(QA_TOKENIZER_PATH)

qa_model = pipeline(
    "question-answering",
    model=model,
    tokenizer=tokenizer
)

Device set to use cpu


In [226]:
def extract_salary_range(desc: str) -> Optional[Tuple]:
    """
    params: the job description as a string
    returns: an tuple of the salary range but it would return None if it wansn't found
    This function aims to eliminate "false positives", it's ok to have some true negatives
    
    NOTE: this function doesn't work well with job description from any language but 
    English
    """
    if not(isinstance(desc, str)):
        return None

    question = "What is the salary range for the role?"
    output = qa_model(question = question, context = desc)
    # print(output)

    if output["score"] < 0.3: 
        return None

    # TODO: handle differnt ways of salary/wage formating
    matches = re.findall(
        r"[\$|€|-| ]([0-9,.k]+)",
        output["answer"],
        flags=re.IGNORECASE
    )
    # print(matches)
    
    if not(matches):
        return None
    if len(matches) > 2:
        matches = matches[:2]
    elif len(matches) == 1:
        matches = [matches[0], matches[0]]
    
    salary_range: list = [None, None] # will be converted into a tuple
    
    for i in range(2):
        is_hour_rate: bool = False
        mag: float = 1.0

        if "," in matches[i]:
            if len(matches[i].split(",")[1]) == 2:
                matches[i] = matches[i].replace(",", ".")
                is_hour_rate = True

        if "k" in matches[i].lower():
            mag = 1000.0

        matches[i] = matches[i].lower()
        matches[i] = matches[i].replace(",", "")
        matches[i] = matches[i].replace("k", "")

        salary_range[i] = float(matches[i]) * mag

    return tuple(salary_range)

In [225]:
%%timeit
extract_salary_range(linkedin_df["description"].sample().iloc[0])

{'score': 8.128779711569223e-06, 'start': 1333, 'end': 1359, 'answer': 'Bachelors degree or higher'}
{'score': 1.0095571269630454e-05, 'start': 1704, 'end': 1717, 'answer': 'PhD is a plus'}
{'score': 0.0003246641313694454, 'start': 1178, 'end': 1185, 'answer': '60+ WPM'}
{'score': 7.909673513495363e-06, 'start': 1562, 'end': 1630, 'answer': '\nGreek speaker (candidates based in Athens will be considered a plus'}
{'score': 0.357524573802948, 'start': 2297, 'end': 2310, 'answer': '$10K annually'}
['10K']
{'score': 0.871249794960022, 'start': 1093, 'end': 1107, 'answer': '$65,000–90,000'}
['65,000']
{'score': 0.0006074637640267611, 'start': 2379, 'end': 2406, 'answer': '$150,000 USD - $200,000 USD'}
{'score': 0.002352246316149831, 'start': 941, 'end': 943, 'answer': '40'}
The slowest run took 7.97 times longer than the fastest. This could mean that an intermediate result is being cached.
773 ms ± 480 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [227]:
salary_ranges = linkedin_df["description"].apply(
    lambda desc: extract_salary_range(desc) if extract_salary_range(desc) else None
)

linkedin_df["salary_min"] = salary_ranges.apply(lambda x: list(x)[0] if x is not None else x)
linkedin_df["salary_max"] = salary_ranges.apply(lambda x: list(x)[1] if x is not None else x)

In [228]:
len(linkedin_df["salary_min"]) - linkedin_df["salary_min"].isnull().sum()

np.int64(232)

Now I need to seperate hour pays from annaul salaries

In [229]:
linkedin_df["pay_type"] = linkedin_df["salary_min"].apply(
    lambda s: None if np.isnan(s) else "Hourly" if s < 200 else "Annually"
)

Extracting skills from the job descriptions, the skills I am searching for are the ones<br>
those exist in the UpWork skills column, instead of writing every skill I am searching for<br>
manualy.<br>

In [230]:
all_skills = set()
for skills in upwork_df["skills"].apply(ast.literal_eval):
    all_skills.update(skills)

all_skills = list(all_skills)

for i in range(len(all_skills) - 1):
    if all_skills[i] == "":
        del all_skills[i]

    if len(all_skills[i]) == 1:
        all_skills[i] = " " + all_skills[i] + " " # Looking at you R

def extract_skills(desc: str) -> List[str]:
    found_skills = []
    for skill in all_skills:
        if skill.lower() in desc.lower():
            found_skills.append(skill)
            
    return found_skills

In [231]:
linkedin_df["skills"] = linkedin_df["description"].apply(
    lambda s: ",".join(extract_skills(s))
)

In [232]:
linkedin_df[["skills", "salary_min", "pay_type"]].sample(7)

,skills,salary_min,pay_type
468,"Graph Neural Network,Algorithms,Generative AI,...",NaN,None
343,"Tableau,SQL,Report",NaN,None
828,"Invoice,Tableau,Data Analysis, R ,Dashboard,SQ...",NaN,None
384,"Data Modeling,Data Analysis,Google Cloud Platf...",NaN,None
229,"Data Analysis,Generative AI,SQL,Machine Learni...",NaN,None
121,"Data Modeling,Data Analysis,Algorithms,SQL,Mac...",NaN,None
472,"Data Modeling,Kubernetes,Microsoft Azure,Algor...",NaN,None


### UpWork

Fixing the empty strings in the `skills` column

In [233]:
list(upwork_df["skills"].sample(1))

["['Machine Learning Model', 'Natural Language Processing', 'Python', 'Network Analysis', 'Text Analysis', '', '', '', '', '', '', '', '', '']"]

In [234]:
upwork_df["skills"] = upwork_df["skills"].apply(
    lambda list_: str(list(filter(lambda s: len(s) > 0, ast.literal_eval(list_))))
)

In [235]:
list(upwork_df["skills"].sample(1))

["['Data Analysis', 'Data Mining', 'Data Science', 'Python', 'Artificial Neural Network', 'Deep Neural Network']"]

Now let's remove the clutter strings from some of the columns

In [236]:
upwork_df[["hours_worked", "hourly_jobs_done", "fixed_jobs_done"]].sample(5)

,hours_worked,hourly_jobs_done,fixed_jobs_done
484,14 hours worked,4 hourly jobs,18 fixed price jobs
49,1102 hours worked,7 hourly jobs,14 fixed price jobs
612,2 hours worked,1 hourly job,11 fixed price jobs
608,236 hours worked,10 hourly jobs,10 fixed price jobs
407,65 hours worked,5 hourly jobs,64 fixed price jobs


In [237]:
def extract_value(s: str) -> int | None:
    value_pattern = re.compile("[0-9]+")

    if not(isinstance(s, str)):
        return None

    match = value_pattern.search(s)

    if not(match):
        return None

    return int(match.group())

for col in ["hours_worked", "hourly_jobs_done", "fixed_jobs_done"]:
    upwork_df[col] = upwork_df[col].apply(extract_value)

In [238]:
upwork_df["hours_worked"].sample(5)

143    2957.0
462      23.0
228       NaN
90        NaN
12     1753.0
Name: hours_worked, dtype: float64

Converting the money format from being a string into being a float for the `hour_rate` and `earnings`<br>
columns

In [239]:
upwork_df[["earnings", "hour_rate"]].head(5)

,earnings,hour_rate
0,None,$4.8
1,$2K+ earned,$3.5
2,$10K+ earned,$5
3,$100K+ earned,$5
4,None,$5


In [240]:
hour_rate_pattern = re.compile(r"\$[0-9.]+")
earnings_pattern = re.compile(r"\$[0-9]+")

def extract_earnings(s: str) -> int | None:
    if not(isinstance(s, str)):
        return None

    match = earnings_pattern.search(s)
    if not(match):
        return None

    magnitude = 1
    if "K" in s:
        magnitude = 1000
    elif "M" in s:
        magnitude = 1000_000

    return int(match.group()[1:]) * magnitude

def extract_hour_rate(s: str) -> float | None:
    if not(isinstance(s, str)):
        return None

    match = hour_rate_pattern.search(s)

    if not(match):
        return None

    return float(match.group()[1:])

upwork_df["earnings"] = upwork_df["earnings"].apply(extract_earnings)
upwork_df["hour_rate"] = upwork_df["hour_rate"].apply(extract_hour_rate)

In [241]:
upwork_df[["earnings", "hour_rate"]].head(5)

,earnings,hour_rate
0,NaN,4.8
1,2000.0,3.5
2,10000.0,5.0
3,100000.0,5.0
4,NaN,5.0


### Guru

Let's fix the `feedback_percent` and `earnings` format & dtype

In [242]:
guru_df[["feedback_percent", "earnings"]].head(5)

,feedback_percent,earnings
0,None,$0
1,100%,$28K
2,100%,$65K
3,100%,$489K
4,98.8%,$28K


In [243]:
def extract_feedback(s: str) -> float | None:
    pattern = re.compile(r"[0-9.]+")

    if not(isinstance(s, str)):
        return None

    match = pattern.search(s)

    if not(match):
        return None

    return float(match.group())

def extract_earnings(s: str) -> int | None:
    pattern = re.compile(r"\$[0-9,]+")

    if not(isinstance(s, str)):
        return None

    match = pattern.search(s)

    if not(match):
        return None

    magnitude = 1
    if "K" in s:
        magnitude = 1000
    elif "M" in s:
        magnitude = 1000_000
    
    earnings_str = match.group()[1:].replace(",", ".")

    return float(earnings_str) * magnitude

guru_df["feedback_percent"] = guru_df["feedback_percent"].apply(extract_feedback)
guru_df["earnings"] = guru_df["earnings"].apply(extract_earnings)

In [244]:
guru_df[["feedback_percent", "earnings"]].head(5)

,feedback_percent,earnings
0,NaN,0.0
1,100.0,28000.0
2,100.0,65000.0
3,100.0,489000.0
4,98.8,28000.0


# Data storing

In [246]:
with sqlite3.connect(CLEAN_DB_PATH) as con:
    linkedin_df.to_sql("linkedin", con, if_exists="fail", index=False)
    upwork_df.to_sql("upwork", con, if_exists="fail", index=False)
    guru_df.to_sql("guru", con, if_exists="fail", index=False)